# Model Testing Notebook

Evaluate and compare different model configurations:
- Whisper model sizes (tiny vs base vs small vs medium)
- LLM backends (OpenAI GPT-3.5 vs GPT-4 vs HuggingFace BART)
- Embedding models for RAG

In [ ]:
import sys, time
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

SAMPLE_TEXT = """
Machine learning is a branch of artificial intelligence that enables systems
to learn from data. Deep learning uses neural networks with multiple layers.
Natural language processing allows computers to understand human language.
We need to review the model architecture next week and assign tasks to the team.
"""

print('Setup complete ✅')

## 1. Compare Whisper Models

In [ ]:
import whisper, time

AUDIO_FILE = '../data/audio/sample.wav'  # Provide a test WAV file
MODELS     = ['tiny', 'base', 'small']

results = []
for model_name in MODELS:
    print(f'Testing Whisper {model_name}...')
    t0    = time.time()
    model = whisper.load_model(model_name, download_root='../models/whisper')
    out   = model.transcribe(AUDIO_FILE)
    elapsed = time.time() - t0
    results.append({'model': model_name, 'time_s': round(elapsed, 2), 'words': len(out['text'].split())})
    print(f'  → {elapsed:.1f}s | {len(out["text"].split())} words')

import pandas as pd
pd.DataFrame(results)

## 2. Compare Summarization Quality

In [ ]:
# HuggingFace BART
from transformers import pipeline

hf_pipe = pipeline('summarization', model='facebook/bart-large-cnn', device=-1)
hf_out  = hf_pipe(SAMPLE_TEXT, max_length=100, min_length=30, do_sample=False)
print('BART Summary:')
print(hf_out[0]['summary_text'])

In [ ]:
# OpenAI GPT-3.5
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
resp   = client.chat.completions.create(
    model='gpt-3.5-turbo',
    messages=[{
        'role': 'user',
        'content': f'Summarize in 3 bullet points:\n\n{SAMPLE_TEXT}'
    }],
    max_tokens=200,
)
print('GPT-3.5 Summary:')
print(resp.choices[0].message.content)

## 3. Embedding Model Comparison

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, time

MODELS = [
    'all-MiniLM-L6-v2',
    'all-mpnet-base-v2',
    'paraphrase-MiniLM-L3-v2',
]

sentences = [
    'Machine learning is a subset of AI',
    'Deep learning uses neural networks',
    'Natural language processing handles text',
    'Action items should be reviewed weekly',
]

for m_name in MODELS:
    t0    = time.time()
    model = SentenceTransformer(m_name)
    embs  = model.encode(sentences)
    print(f'{m_name}: dim={embs.shape[1]}, time={time.time()-t0:.2f}s')

## 4. RAG Retrieval Quality Test

In [ ]:
from backend.services.rag_pipeline import RAGPipeline

TEST_CHUNKS = [
    {'chunk_id': i, 'text': s, 'start_ts': f'00:0{i}:00', 'end_ts': f'00:0{i+1}:00',
     'start': i*60.0, 'end': (i+1)*60.0}
    for i, s in enumerate(sentences)
]

rag = RAGPipeline()
rag.index_chunks(TEST_CHUNKS)

TEST_QUERIES = [
    'artificial intelligence and machine learning',
    'neural network architecture',
    'weekly tasks and action items',
]

for q in TEST_QUERIES:
    results = rag.query(q, top_k=2)
    print(f'\nQuery: {q}')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["text"]}')